

Este tutorial presenta un ejemplo de pronóstico estacional de caudales utilizando el método de la ***persisténcia hidrológica***. En resumen, este método se basa en la premisa de que la anomalía del caudal observada en el mes más reciente persistirá en los próximos *N* meses del alcance del pronóstico.

Los scripts abajo son basados en la metodología presentada en:



*   *Svensson, C. (2016). Seasonal river flow forecasts for the United Kingdom using persistence and historical analogues. Hydrological Sciences Journal, 61(1), 19-35.*

---

# **Clonando el repositório del HydroSOS**

Clonaremos el repositório del WMO HydroSOS hospedado en Github para colectar una serie temporal ejemplo de caudales diarios.

*   Obs: Para utilizar el comando `git clone` en alguna  plataforma de ejecución de notebooks/scripts python en loa máquina local (e.g., ***Jupyter***, ***Visual Studio Code***), hay que certificarse de que el `Git` se encuentra instalado en su ordenador. En caso contrário, el git se puede descargar en (https://git-scm.com/downloads)



In [ ]:
# Clone git respository
!git clone https://github.com/wmo-im/HydroSOS/

---

# **Importando librerías python**

Importaremos las librerías necesarias para la realización del tutorial:

In [ ]:
import os
import pandas as pd
import numpy as np

from pathlib import Path
from datetime import datetime

---

# **Creando carpetas**

Crearemos automaticamente una carpeta `output` en el diretorio raiz donde vamos exportar los resultados:


In [ ]:
# Define folder path (e.g., in the current working directory)
folder_output = 'output'

# Create the folder if it doesn't exist
os.makedirs(folder_output, exist_ok=True)

---

# **1. Definición de los parámetros**

Aquí definiremos los parámetros para generación del pronóstico basado en la persistencia hidrológica:

*   `File  path`: Camino donde se encuentra el archivo .csv con los caudales diarios
*   `fcst_range`: Alcance (en meses) del pronóstico pretendido
*   `apply_transformation`: Indica si queremos aplicar transformación logaritmica a los caudales mensuales (True/False)
*   `min_valid_frac`: Fracción mínima de datos diários válidos en un mes para converterlos a promedio mensual

*   `clim_startYear`: Año inicial del período de referéncia para el cálculo de las anomalias
*   `clim_endYear`: Año final del período de referéncia para el cálculo de las anomalias





In [ ]:
file_path = './HydroSOS/example_data/forecast/input/obs_dir/obs_1.csv' # Path to your CSV file
fcst_range = 3                    # Forecast range (months ahead)
apply_transformation = True       # Set to True to apply log transformation on flows
min_valid_frac = 0.5              # Minimum fraction of valid daily data per month

clim_startYear = 1991             # Start year to compute anomalies (mean, standard deviation)
clim_endYear = 2020               # End year to compute anomalies

---

# **2. Lectura de los datos**

2.1. Haremos la lectura del archivo .csv con los datos de caudales diários y los convertiremos a mensuales. El archivo .csv debe tener 2 columnas (una con las fechas, y otra con los valores de caudales), además de:

1.   El titulo de la columna con las fechas definido como '*Date*' (sin aspas)
2.   Las fechas en formato **YYYY-MM-DD**

In [ ]:
# Read and preprocess data
df = pd.read_csv(file_path)
df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d')

df.set_index('Date', inplace=True)

# Convert to monthly data
df_month = df.resample('MS').mean()

2.2. Plotando el gráfico de los caudales:

In [ ]:
#@title Grafico
# Plot on screen (subplot for daily and monthly data)
from matplotlib import pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 6))
axes = axes.flatten()

df.plot(ax=axes[0], color='k', linewidth=0.5, title='Diario')
df_month.plot(ax=axes[1], color='k', linewidth=0.5, title='Mensual')

for ax in axes:
  ax.set_ylabel('Caudal (m³/s)')
  ax.legend(['Observado'])

plt.tight_layout()
plt.show()

---

# **3. Pré-procesamiento de los datos mensuales**


3.1. Removemos los meses con insuficiencia de datos diarios de acuerdo con el mínimo definido en la sección de parámetros:

In [ ]:
# Resample daily to monthly step and compute de fraction of dates in each month without missing data
valid_fraction = df.resample('MS').apply(lambda x: x.count() / x.size)

# Replace monthly values to NaN if % of valid < minimum
df_month.mask(valid_fraction < min_valid_frac, np.nan, inplace=True)

3.2. Aplicando transformación logarítmica $y = log(Q)$ sobre la série de caudales mensuales (si la opción ha sido activada por el usuario en la definición de los parámetros). Este proceso debería dejar la distribuición de los caudales más similar a la normal para el cálculo de las anomalías mensuales estandarizadas:

In [ ]:
# Apply log transformation if selected
if apply_transformation:
    if (df_month <= 0).sum().values > 0:
        raise ValueError("Log transformation requires all discharge values to be > 0.")
    df_month_transformed = np.log(df_month)
else:
    df_month_transformed = df_month.copy()

3.3. Podemos verificar el histograma de los caudales para ver como está la distribuición:

In [ ]:
# Plot the discharge distribution in a histogram
df_month_transformed.hist()

---

# **4. Cálculo de anomalías estándares**



4.1. Cálculo del promedio $ (\overline{x}_m) $ y desvio estándar $ (\sigma_m) $ referente a cada mes del calendario. Las estadísticas son calculadas para el período climatológico de referéncia:

In [ ]:
# Converting date strings to datetime format
dt_start = datetime(clim_startYear, 1, 1)
dt_end = datetime(clim_endYear, 12, 31)

# Get data for the period of interest
monthly_transformed_clim = df_month_transformed[(df_month_transformed.index >= dt_start) &
                                                  (df_month_transformed.index <= dt_end)]

# Compute mean and standard deviation for each calendar month
month_nums = monthly_transformed_clim.index.month
clim_mean = monthly_transformed_clim.groupby(month_nums).mean()
clim_std = monthly_transformed_clim.groupby(month_nums).std()

4.2. Plotando los valores de promedio y desvio estándar:

In [ ]:
# For visualization purposes only
conc_data = pd.concat([clim_mean, clim_std], axis=1)
conc_data.columns = ['Promedio', 'Desvio std']
conc_data.index.name = 'Mes'
display(conc_data)

4.3. Cálculo de anomalias estandarizadas para cada valor de la serie mensual. El cálculo utiliza el promedio y desvio estándar del mes correspondiente al valor observado, conforme la siguiente equación:  

\begin{align}
  a_{t,m} = \frac{q_{t,m}-\overline{x_m}}{\sigma_m}
\end{align}

Donde:

*   $ a_{t,m} $ es la anomalía para el intervalo de tiempo $t$ y mes $m$;
*   $ q_{t,m} $ es caudal (transformado) para el intervalo de tiempo $t$ y mes $m$.



In [ ]:
# Compute standardized anomalies
df_anomalies = df_month_transformed.copy()

# Compute anomalies for each month separately
for i in range(1, 13):
  idx = (df_anomalies.index.month == i)
  df_anomalies.loc[idx] = (df_month_transformed.loc[idx] - clim_mean.loc[i].values) / clim_std.loc[i].values

4.4. Plotando el gráfico de las anomalias estandarizadas para toda la serie mensual:

In [ ]:
#@title Grafico
# Plot on screen
df_anomalies.plot(figsize=(12, 4),
                  title=f'Anomalias mensuales estandarizadas, referéncia: {clim_startYear}-{clim_endYear}',
                  ylabel='Anomalia (-)',
                  legend=False)

---

# **5. Generación del pronóstico**

5.1. Para generar el pronóstico primeramente identificaremos la anomalía más reciente observada:

In [ ]:
# Check standardized anomaly for last month
anomaly_date = df_month.index[-1]
anomaly_value = df_anomalies.loc[anomaly_date].values

print(f'Mes de la anomalía: {anomaly_date.strftime("%Y-%m")}')
print(f'Anomalía verificada: {anomaly_value}')

5.2. Identificamos las fechas de los *N* meses del pronóstico y persistimos la anomalía verificada sobre el promedio y desvio estándar de cada uno de ellos, a través de la equación:  

\begin{align}
  q_m = \overline{x_m} + (a_o \times \sigma_m)
\end{align}

Donde:

*   $ a_{o} $ es la anomalía observada en el mes más reciente;



In [ ]:
# Generate forecast dates
forecast_dates = pd.date_range(start=anomaly_date + pd.offsets.MonthEnd(1),
                               periods=fcst_range,
                               freq='MS')

# Get forecast months
forecast_month_nums = forecast_dates.month

# Get the climatological mean and standard deviation for these months
clim_mean_fcst = clim_mean.loc[forecast_month_nums].values
clim_std_fcst = clim_std.loc[forecast_month_nums].values

# Reconstruct forecast values by persisting anomaly on climatological mean and standard deviation
df_forecast_transformed = clim_mean_fcst + (anomaly_value * clim_std_fcst)

5.3. Transformación de los caudales de vuelta al espacio real $ ( Q = e^y ) $ (en caso que hayan sido transformados anteriormente):

In [ ]:
# Back-transform if log transformation was applied
if apply_transformation:
    forecast_values = np.exp(df_forecast_transformed)
else:
    forecast_values = df_forecast_transformed

# Ensure only positive values
forecast_values[forecast_values < 0.0] = 0.0

5.4. Creación de la tabla final con las fechas y valores obtenidos para el pronóstico:

In [ ]:
# Output forecast
df_forecast = pd.DataFrame(forecast_values,
                           index = forecast_dates,
                           columns = df_month.columns)

df_forecast.index.name = 'Date'

5.5. Plotando el gráfico con los pronósticos generados:

In [ ]:
#@title Grafico
# Plot on screen
fig, ax = plt.subplots(figsize=(15, 4))

df_month[-36:].plot(ax=ax, color='k') #Plotting the last 36 months
df_forecast.plot(ax=ax, color='r')
ax.set_ylabel('Caudal (m³/s)')
ax.set_title('Pronóstico estacional basado en la persisténcia')

plt.legend(['Observado', 'Pronóstico'])
plt.show()

# **6. Exportando los pronósticos generados**

Finalmente exportamos los resultados a un archivo .csv:

In [ ]:
# Save to CSV
file_name = Path(file_path).stem
df_forecast.to_csv(f'{folder_output}/persist_forecast_{file_name}.csv', float_format='%.2f')